In [ ]:
import os
import random
import pickle
import joblib
import json
import time
from collections import deque
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init
import torch.nn.functional as F
from torch.autograd import Variable
import matplotlib.pyplot as plt
import warnings
from torch.utils.data import Dataset, DataLoader, TensorDataset, WeightedRandomSampler
from tqdm.notebook import tqdm

from sklearn import datasets, linear_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.metrics import f1_score, roc_auc_score
from ast import literal_eval
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from pytorch_metric_learning import losses
from torcheval.metrics.functional import multiclass_f1_score
from torcheval.metrics.functional.classification import multiclass_recall
from torcheval.metrics.functional import multiclass_accuracy
from torcheval.metrics.functional import multiclass_auprc
from torcheval.metrics.functional import multiclass_auroc

pd.set_option('display.max_seq_items', None)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

SEED = 0
seed_everything(seed = SEED)

In [ ]:
class Encoder(nn.Module): #supervised contrastive learning
    def __init__(self, input_dim, hidden_dims, drop_rate):
        super(Encoder, self).__init__()
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        
        leaky_relu = nn.LeakyReLU()
        
        encoder = [nn.Linear(input_dim, hidden_dims[0]),
                   nn.BatchNorm1d(hidden_dims[0]),
                   nn.Dropout(drop_rate),
                   leaky_relu]
    
        
        for i in range(1,len(hidden_dims)):
        
                encoder.append(nn.Linear(hidden_dims[i-1], hidden_dims[i]))
                encoder.append(nn.BatchNorm1d(hidden_dims[i]))
                encoder.append(nn.Dropout(drop_rate))
                encoder.append(leaky_relu)
                
        self.encoder = nn.Sequential(
            *encoder
        )
        

    def forward(self, x):
        latent = self.encoder(x)
        return latent
    
class MultiTaskModel(nn.Module):
    def __init__(self, encoder, latent_dim, main_output_dim, sub1_output_dim, sub2_output_dim, sub3_output_dim): #, sub4_output_dim
        super(MultiTaskModel, self).__init__()
        self.encoder = encoder
        self.sub_head1 = nn.Linear(latent_dim, sub1_output_dim)
        self.sub_head2 = nn.Linear(latent_dim, sub2_output_dim)
        self.sub_head3 = nn.Linear(latent_dim, sub3_output_dim)
        #self.sub_head4 = nn.Linear(latent_dim, sub4_output_dim)
    
    def forward(self, x):
        latent = self.encoder(x)
        sub1_output = self.sub_head1(latent)
        sub2_output = self.sub_head2(latent)
        sub3_output = self.sub_head3(latent)
        #sub4_output = self.sub_head4(latent)

        return latent, sub1_output, sub2_output, sub3_output #, sub4_output
    
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Cross entropy loss
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)  # Probabilities of the true class
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
def param():
    USER_NAME = ''                  
    SEED      = 0
    batch_size= 512
    epoch     = 120
    lr        = 0
    drop_rate = 0.223573505016481
    hidden = {
        'hidden' : []
     }
    temp = 0.3137977865167101
    return hidden, SEED, batch_size, lr, epoch, drop_rate, temp

def load_data(df):
    x = df[['Heart rate', 'Blood pressure systolic', 'Blood pressure diastolic', 'Blood pressure mean', 'Respiratory rate', 'SpO2', 'Temperature', 'Shock index',
            'Heart rate:Delta', 'Blood pressure systolic:Delta', 'Blood pressure diastolic:Delta', 'Blood pressure mean:Delta', 'Respiratory rate:Delta', 'SpO2:Delta', 'Temperature:Delta', 'Shock index:Delta',
            'Lactate', 'Creatinine', 'Hemoglobin',  'Troponin', 'Lactate Elevation index', 'Creatinine Elevation index', 'Hemoglobin Elevation index', 
            'Propofol', 'Midazolam', 'Fentanyl', 'Ketamine', 'Treated_Diuretic:Binary']]
    
    fn = x.shape[1]
    print(f'{fn} Features: ', x.columns)
    x['Ketamine'] = x['Ketamine'].astype('category')
    x['Propofol'] = x['Propofol'].astype('category')
    x['Midazolam'] = x['Midazolam'].astype('category')
    x['Fentanyl'] = x['Fentanyl'].astype('category')
    x['Hemoglobin Elevation index'] = x['Hemoglobin Elevation index'].astype('category')
    x['Creatinine Elevation index'] = x['Creatinine Elevation index'].astype('category')
    x['Lactate Elevation index'] = x['Lactate Elevation index'].astype('category')

    y_main = df['Cont_label'].astype('category')
    y_sub1 = df['MAP_Status']
    y_sub2 = df['Shock']
    y_sub3 = df['SCL_Target']
    
    return x, y_main, y_sub1, y_sub2, y_sub3


def train(trial, search=True):
    global emb_model, results_loss, optimizer, scheduler
    
    hidden, SEED, batch_size, lr, epoch, drop_rate, temp = param()
    
    # search parameters
    if search == True:
        for idx, i in enumerate(hidden):
            num_layers = trial.suggest_int(f'num_layer_{idx}', 3, 10)
            for i in range(num_layers-1):
                hidden['hidden'].append(trial.suggest_int(f'h{i+1}', 200, 500))
            hidden['hidden'].append(trial.suggest_int(f'h{num_layers}', 50, 200))
        epoch     = 100
        lr        = trial.suggest_categorical('Learning_rate',[0.0005])
        drop_rate = 0
        temp      = trial.suggest_uniform('temp', 0.1, 0.5)

    # hidden['hidden'] = sorted(hidden['hidden'], reverse=True)       
    print(hidden)
    print('learning_rate : ', lr, "\nepoch : ", epoch, "\ndrop_rate : ", drop_rate, "\ntemperature : ", temp)
    
    results_loss = {
    'epoch_by_total'          : [],
    'epoch_by_val'          : [],
    'epoch_by_main_cont'          : [],
    'epoch_by_sub1_CE'          : [],
    'epoch_by_sub2_CE'          : [],
    'epoch_by_sub3_CE'          : [],
    'epoch_by_sub4_CE'          : []
    }

    mimic_df = dataset.copy()
    trn_x, trn_y_main, trn_y_sub1, trn_y_sub2, trn_y_sub3 = load_data(mimic_df)
    
    scaler = MinMaxScaler()
    trn_scaled_x = scaler.fit_transform(trn_x)
    trn_scaled_x[np.isnan(trn_scaled_x)] = -2

    trn_tensor_x = torch.FloatTensor(trn_scaled_x)
    trn_tensor_y_main = torch.LongTensor(trn_y_main.values)
    trn_tensor_y_sub1 = torch.FloatTensor(trn_y_sub1.values)
    trn_tensor_y_sub2 = torch.FloatTensor(trn_y_sub2.values)
    trn_tensor_y_sub3 = torch.FloatTensor(trn_y_sub3.values)

    train_dataset = TensorDataset(trn_tensor_x, trn_tensor_y_main, trn_tensor_y_sub1, trn_tensor_y_sub2, trn_tensor_y_sub3)
    
    y_train_indices = mimic_df.index

    y_train = [mimic_df['Cont_label'][i] for i in y_train_indices]


    class_sample_count = np.array(
        [len(np.where(y_train == t)[0]) for t in np.unique(y_train)])
    

    weight = 1. / class_sample_count
    
    samples_weight = np.array([weight[t] for t in y_train])
    samples_weight = torch.from_numpy(samples_weight)
    
    sampler = WeightedRandomSampler(samples_weight.type('torch.DoubleTensor'), len(samples_weight))

    train_loader  = torch.utils.data.DataLoader(dataset= train_dataset, batch_size=batch_size, shuffle=False, sampler=sampler, drop_last=True)

    n_feat = trn_tensor_x.shape[1]
    encoder = Encoder(n_feat, hidden['hidden'], drop_rate).to(device)
    emb_model = MultiTaskModel(encoder, hidden['hidden'][-1], main_output_dim=1, sub1_output_dim=1, sub2_output_dim=1, sub3_output_dim=1).to(device)
    
    contrastive_loss = losses.SupConLoss(temperature=temp)
    sub_loss_fn = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss() 
    focal_loss_fn = FocalLoss(alpha=1, gamma=2)
    optimizer = optim.RMSprop(emb_model.parameters(), lr=lr)
    scheduler = CosineAnnealingWarmUpRestarts(optimizer, T_0=50, T_mult=1, eta_max=0.0001, T_up=50, gamma=0.5)
    
    patience_value = 0
    patience = 10
    for i in tqdm(range(1, epoch+1)):
        emb_model.train()
        
        train_loss = {'total_loss': [], 'main_cont_loss': [], 'sub1_CE_loss': [], 'sub2_CE_loss': [], 'sub3_CE_loss': []} #, 'sub4_CE_loss': []
        
        start = time.time()
        
        for j, (X, y_main, y_sub1, y_sub2, y_sub3)  in enumerate(train_loader):
            
            X = X.to(device)
            y_main = y_main.to(device)
            y_sub1 = y_sub1.to(device)
            y_sub2 = y_sub2.to(device)
            y_sub3 = y_sub3.to(device)

            optimizer.zero_grad()
            
            latent, sub1_output, sub2_output, sub3_output = emb_model(X)
            
            y_main = y_main.long()
            y_sub1 = y_sub1.float()
            y_sub2 = y_sub2.float()
            y_sub3 = y_sub3.float()

            loss_cont_main = contrastive_loss(latent, y_main)
            loss_CE_sub1 = mse_loss(sub1_output, y_sub1)
            loss_CE_sub2 = mse_loss(sub2_output, y_sub2)
            loss_CE_sub3 = mse_loss(sub3_output, y_sub3)
            
            total_loss = loss_cont_main + 0.5*(loss_CE_sub1 + loss_CE_sub2+ loss_CE_sub3)
            
            total_loss.backward()
            optimizer.step()
            scheduler.step() 
            
            train_loss['total_loss'].append(total_loss.item())
            train_loss['main_cont_loss'].append(loss_cont_main.item())
            train_loss['sub1_CE_loss'].append(loss_CE_sub1.item())
            train_loss['sub2_CE_loss'].append(loss_CE_sub2.item())
            train_loss['sub3_CE_loss'].append(loss_CE_sub3.item())

        total_loss_mean = np.array(train_loss['total_loss']).mean()
        main_cont_loss_mean = np.array(train_loss['main_cont_loss']).mean()
        sub1_CE_loss_mean = np.array(train_loss['sub1_CE_loss']).mean()
        sub2_CE_loss_mean = np.array(train_loss['sub2_CE_loss']).mean()
        sub3_CE_loss_mean = np.array(train_loss['sub3_CE_loss']).mean()

        results_loss['epoch_by_total'].append(total_loss_mean)
        results_loss['epoch_by_main_cont'].append(main_cont_loss_mean)
        results_loss['epoch_by_sub1_CE'].append(sub1_CE_loss_mean)
        results_loss['epoch_by_sub2_CE'].append(sub2_CE_loss_mean)
        results_loss['epoch_by_sub3_CE'].append(sub3_CE_loss_mean)
        
        end = time.time()
        if i % 1 == 0:
            print(f'epoch {i}  time: {end - start:.4f}sec total_loss: {total_loss_mean: .4f}')

        if i > 2:
            if results_loss['epoch_by_total'][-1] > results_loss['epoch_by_total'][-2]:
                patience_value += 1

        if patience_value == patience :
            print('------------------------------------------------')
            print(f'epoch {i} End')
            print('================================================')
            patience_value = 0
            break
    
    return total_loss_mean

def train_searched_params(num_layers, node_ls, temperature, learning_rate):
    global emb_model, results_loss, optimizer, scheduler
    
    hidden, SEED, batch_size, lr, epoch, drop_rate, temp = param()
    
    # search parameters
    num_layers = num_layers
    hidden['hidden'] = node_ls
    
    epoch = 500
    lr = learning_rate
    drop_rate = 0
    temp = temperature
    
    print(hidden)
    print('learning_rate : ', lr, "\nepoch : ", epoch, "\ndrop_rate : ", drop_rate, "\ntemperature : ", temp)
    
    results_loss = {
    'epoch_by_total'          : [],
    'epoch_by_val'          : [],
    'epoch_by_main_cont'          : [],
    'epoch_by_sub1_CE'          : [],
    'epoch_by_sub2_CE'          : [],
    'epoch_by_sub3_CE'          : [],
    }

    mimic_df = dataset.copy()
    trn_x, trn_y_main, trn_y_sub1, trn_y_sub2, trn_y_sub3 = load_data(mimic_df)
    
    scaler = MinMaxScaler()
    trn_scaled_x = scaler.fit_transform(trn_x)
    trn_scaled_x[np.isnan(trn_scaled_x)] = -2
    
    trn_tensor_x = torch.FloatTensor(trn_scaled_x)
    trn_tensor_y_main = torch.LongTensor(trn_y_main.values)
    trn_tensor_y_sub1 = torch.FloatTensor(trn_y_sub1.values)
    trn_tensor_y_sub2 = torch.FloatTensor(trn_y_sub2.values)
    trn_tensor_y_sub3 = torch.FloatTensor(trn_y_sub3.values)

    train_dataset = TensorDataset(trn_tensor_x, trn_tensor_y_main, trn_tensor_y_sub1, trn_tensor_y_sub2, trn_tensor_y_sub3)
    
    y_train_indices = mimic_df.index

    y_train = [mimic_df['Cont_label'][i] for i in y_train_indices]


    class_sample_count = np.array(
        [len(np.where(y_train == t)[0]) for t in np.unique(y_train)])
    
    weight = 1. / class_sample_count
    
    samples_weight = np.array([weight[t] for t in y_train])
    samples_weight = torch.from_numpy(samples_weight)

    sampler = WeightedRandomSampler(samples_weight.type('torch.DoubleTensor'), len(samples_weight))

    train_loader  = torch.utils.data.DataLoader(dataset= train_dataset, batch_size=batch_size, shuffle=False, sampler=sampler, drop_last=True)

    # 모델 정의
    n_feat = trn_tensor_x.shape[1]
    encoder = Encoder(n_feat, hidden['hidden'], drop_rate).to(device)
    emb_model = MultiTaskModel(encoder, hidden['hidden'][-1], main_output_dim=1, sub1_output_dim=1, sub2_output_dim=1, sub3_output_dim=1).to(device) #, sub4_output_dim=4
    
    contrastive_loss = losses.SupConLoss(temperature=temp)
    sub_loss_fn = nn.CrossEntropyLoss()
    focal_loss_fn = FocalLoss(alpha=1, gamma=2)
    mse_loss = nn.MSELoss() 
    optimizer = optim.RMSprop(emb_model.parameters(), lr=lr)
    scheduler = CosineAnnealingWarmUpRestarts(optimizer, T_0=50, T_mult=1, eta_max=0.0001, T_up=50, gamma=0.5)
    
    patience_value = 0
    patience = 20
    for i in tqdm(range(1, epoch+1)):
        emb_model.train()
        
        train_loss = {'total_loss': [], 'main_cont_loss': [], 'sub1_CE_loss': [], 'sub2_CE_loss': [], 'sub3_CE_loss': []} #, 'sub4_CE_loss': []
        
        start = time.time()
        
        for j, (X, y_main, y_sub1, y_sub2, y_sub3)  in enumerate(train_loader):
            
            X = X.to(device)
            y_main = y_main.to(device)
            y_sub1 = y_sub1.to(device)
            y_sub2 = y_sub2.to(device)
            y_sub3 = y_sub3.to(device)

            optimizer.zero_grad()
            
            latent, sub1_output, sub2_output, sub3_output = emb_model(X)
            
            y_main = y_main.long()
            y_sub1 = y_sub1.float()
            y_sub2 = y_sub2.float()
            y_sub3 = y_sub3.float()

            loss_cont_main = contrastive_loss(latent, y_main)
            loss_CE_sub1 = mse_loss(sub1_output, y_sub1)
            loss_CE_sub2 = mse_loss(sub2_output, y_sub2)
            loss_CE_sub3 = mse_loss(sub3_output, y_sub3)

            total_loss = loss_cont_main + 0.5*(loss_CE_sub1 + loss_CE_sub2+ loss_CE_sub3)
            
            total_loss.backward()
            optimizer.step()
            scheduler.step() 
            
            train_loss['total_loss'].append(total_loss.item())
            train_loss['main_cont_loss'].append(loss_cont_main.item())
            train_loss['sub1_CE_loss'].append(loss_CE_sub1.item())
            train_loss['sub2_CE_loss'].append(loss_CE_sub2.item())
            train_loss['sub3_CE_loss'].append(loss_CE_sub3.item())

        total_loss_mean = np.array(train_loss['total_loss']).mean()
        main_cont_loss_mean = np.array(train_loss['main_cont_loss']).mean()
        sub1_CE_loss_mean = np.array(train_loss['sub1_CE_loss']).mean()
        sub2_CE_loss_mean = np.array(train_loss['sub2_CE_loss']).mean()
        sub3_CE_loss_mean = np.array(train_loss['sub3_CE_loss']).mean()

        results_loss['epoch_by_total'].append(total_loss_mean)
        results_loss['epoch_by_main_cont'].append(main_cont_loss_mean)
        results_loss['epoch_by_sub1_CE'].append(sub1_CE_loss_mean)
        results_loss['epoch_by_sub2_CE'].append(sub2_CE_loss_mean)
        results_loss['epoch_by_sub3_CE'].append(sub3_CE_loss_mean)
        
        end = time.time()
        if i % 1 == 0:
            print(f'epoch {i}  time: {end - start:.4f}sec total_loss: {total_loss_mean: .4f}')

        if i > 10:
            if results_loss['epoch_by_total'][-1] == min(results_loss['epoch_by_total']):
                torch.save({"model_state_dict": emb_model, "learning_rate": lr, "epoch": i, "layer_node": hidden['hidden']},
                           f"SCL_model.pt")
                patience_value = 0
                
            else:
                patience_value += 1

        if patience_value == patience :
            print('------------------------------------------------')
            print(f'epoch {i} End')
            print('================================================')
            patience_value = 0
            break
    
    return total_loss_mean, i

In [ ]:
def make_embeded_df(model_name): 
    print()
    print('Start Getting the latent space vector(Train, Valid sample)')
    
    mimic_df = dataset.copy()
    print(len(mimic_df))
    
    trn_x, trn_y_main, trn_y_sub1, trn_y_sub2, trn_y_sub3 = load_data(mimic_df)
    
    scaler = MinMaxScaler()
    trn_scaled_x = scaler.fit_transform(trn_x)
    trn_scaled_x[np.isnan(trn_scaled_x)] = -10

    trn_tensor_x = torch.FloatTensor(trn_scaled_x)
    trn_tensor_y_main = torch.LongTensor(trn_y_main.values)
    
    
    n_feat = trn_tensor_x.shape[1]
    train_dataset = TensorDataset(trn_tensor_x, trn_tensor_y_main)
    for_latent_loader_trn  = torch.utils.data.DataLoader(dataset= train_dataset, batch_size=trn_tensor_x.shape[0], shuffle=False, drop_last=False)
    
    start = time.time()
    model_name.eval()
    with torch.no_grad():
        for X_l, y_l  in for_latent_loader_trn:
                
                X_l  = X_l.to(device)
                latent_vector_train, _, _, _ = model_name.forward(X_l)
                
                emb_train_x = pd.DataFrame(np.array(latent_vector_train.cpu()))
                emb_train = pd.concat([emb_train_x, pd.DataFrame(np.array(y_l))], axis = 1)

    end = time.time()            
    print()
    print('End, Time consume(min):{}'.format((end - start)/60))  
    
    return emb_train

In [ ]:
import math
from torch.optim.lr_scheduler import _LRScheduler

class CosineAnnealingWarmUpRestarts(_LRScheduler):
    def __init__(self, optimizer, T_0, T_mult=1, eta_max=0.1, T_up=0, gamma=1., last_epoch=-1):
        if T_0 <= 0 or not isinstance(T_0, int):
            raise ValueError("Expected positive integer T_0, but got {}".format(T_0))
        if T_mult < 1 or not isinstance(T_mult, int):
            raise ValueError("Expected integer T_mult >= 1, but got {}".format(T_mult))
        if T_up < 0 or not isinstance(T_up, int):
            raise ValueError("Expected positive integer T_up, but got {}".format(T_up))
        self.T_0 = T_0
        self.T_mult = T_mult
        self.base_eta_max = eta_max
        self.eta_max = eta_max
        self.T_up = T_up
        self.T_i = T_0
        self.gamma = gamma
        self.cycle = 0
        self.T_cur = last_epoch
        super(CosineAnnealingWarmUpRestarts, self).__init__(optimizer, last_epoch)
    
    def get_lr(self):
        if self.T_cur == -1:
            return self.base_lrs
        elif self.T_cur < self.T_up:
            return [(self.eta_max - base_lr)*self.T_cur / self.T_up + base_lr for base_lr in self.base_lrs]
        else:
            return [base_lr + (self.eta_max - base_lr) * (1 + math.cos(math.pi * (self.T_cur-self.T_up) / (self.T_i - self.T_up))) / 2
                    for base_lr in self.base_lrs]

    def step(self, epoch=None):
        if epoch is None:
            epoch = self.last_epoch + 1
            self.T_cur = self.T_cur + 1
            if self.T_cur >= self.T_i:
                self.cycle += 1
                self.T_cur = self.T_cur - self.T_i
                self.T_i = (self.T_i - self.T_up) * self.T_mult + self.T_up
        else:
            if epoch >= self.T_0:
                if self.T_mult == 1:
                    self.T_cur = epoch % self.T_0
                    self.cycle = epoch // self.T_0
                else:
                    n = int(math.log((epoch / self.T_0 * (self.T_mult - 1) + 1), self.T_mult))
                    self.cycle = n
                    self.T_cur = epoch - self.T_0 * (self.T_mult ** n - 1) / (self.T_mult - 1)
                    self.T_i = self.T_0 * self.T_mult ** (n)
            else:
                self.T_i = self.T_0
                self.T_cur = epoch
                
        self.eta_max = self.base_eta_max * (self.gamma**self.cycle)
        self.last_epoch = math.floor(epoch)
        for param_group, lr in zip(self.optimizer.param_groups, self.get_lr()):
            param_group['lr'] = lr

In [ ]:
warnings.filterwarnings("ignore")
import os
import optuna

# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= '0'
os.environ['CUDA_LAUNCH_BLOCKING']= '1'
n_gpu             = 1
device = torch.device('cpu')
print(device)

study = optuna.create_study(sampler=optuna.samplers.TPESampler(), direction="minimize")
study.optimize(train, n_trials = 10)  

pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

Parameter 정보 꼭 기록하기

In [ ]:
warnings.filterwarnings("ignore")
import os
import optuna

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= '0'
os.environ['CUDA_LAUNCH_BLOCKING']= '1'
n_gpu             = 1
device = torch.device('cpu')
print(device)

version = 'ver_Final'

model_loss, epoch_now = train_searched_params(num_layers=7, node_ls=[216, 320, 289, 259, 320, 327, 147] , temperature=0.14835399159684926, learning_rate=5e-04)

In [ ]:
lrs = []
for i in range(120):
    optimizer.step()
    lrs.append(optimizer.param_groups[0]["lr"])
    scheduler.step()


plt.plot(range(120), lrs, color = 'limegreen',  label = 'Training Cosine Annealing Warm Restarts')
plt.xlabel('Epoch')
plt.ylabel('Learning rate(green)')
plt.legend()
plt.show()

In [ ]:
def load_trained_model_full_object(model_path, device='cpu'):
    # 전체 모델 객체 로드
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)

    
    # 모델 객체 추출 (dict로 저장된 경우)
    model = checkpoint['model_state_dict']  # 모델 객체가 dict 안에 'model' 키로 저장되었다고 가정
    
    print("전체 모델 객체가 성공적으로 로드되었습니다!")
    return model

def save_state_dict_from_full_model(model, save_path):
    # 전체 모델에서 state_dict 추출
    state_dict = model.state_dict()
    
    # state_dict를 파일로 저장
    torch.save({
        'model_state_dict': state_dict
    }, save_path)
    print(f"state_dict가 {save_path}에 저장되었습니다.")

# 모델 경로
model_path = f"SCL_model.pt"
emb_model_load = load_trained_model_full_object(model_path, device)

# state_dict를 저장할 경로
state_dict_path = f"SCL_model.pth"
save_state_dict_from_full_model(emb_model_load, state_dict_path)


In [ ]:
def load_trained_model(model_path, n_features, hidden_layers, drop_rate, main_output_dim, sub1_output_dim, sub2_output_dim, sub3_output_dim, device='cpu'):
    # 모델 아키텍처 정의
    encoder = Encoder(n_features, hidden_layers, drop_rate).to(device)
    model = MultiTaskModel(encoder, hidden_layers[-1], 
                           main_output_dim=main_output_dim, 
                           sub1_output_dim=sub1_output_dim,
                           sub2_output_dim=sub2_output_dim,
                           sub3_output_dim=sub3_output_dim).to(device)

    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        print(f"Model loaded successfully! Epoch: {checkpoint.get('epoch')}, Loss: {checkpoint.get('loss')}")
    else:
        raise ValueError("Invalid checkpoint format. Expected a dict with 'model_state_dict'.")
    
    model.eval()
    return model

# 모델 경로 및 파라미터 정의
model_path = "SCL_model.pth"
n_features = 28
hidden_layers = [216, 320, 289, 259, 320, 327, 147]
drop_rate = 0
main_output_dim = 1
sub1_output_dim = 1
sub2_output_dim = 1
sub3_output_dim = 1

# 모델 로드
emb_model_load = load_trained_model(model_path, n_features, hidden_layers, drop_rate, 
                                    main_output_dim, sub1_output_dim, sub2_output_dim, sub3_output_dim, device='cpu')


In [ ]:
emb_df = make_embeded_df(emb_model_load)
sample = pd.concat([dataset.reset_index(drop=True), emb_df.reset_index(drop=True)], axis = 1, ignore_index=False)
sample.to_csv(f'', index=False)
sample = pd.read_csv(f'')
sample['Cont_label'] = sample['0.1']
sample.drop(columns=['0.1'], inplace=True)
sample.to_csv(f'', index=False)